# Delta table Query util
DuckDB natively supports reading Delta tables using `delta_scan()`.

In [1]:
import os
import certifi
import duckdb
from electricity_maps.config import get_settings

os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()


get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir

In [2]:
import re

_con = None

def get_duckdb_connection():
    global _con
    if _con is not None:
        return _con

    _con = duckdb.connect()
    _con.execute("INSTALL delta;")
    _con.execute("LOAD delta;")

    if str(DATA_DIR).startswith('s3://'):
        _con.execute("INSTALL httpfs;")
        _con.execute("LOAD httpfs;")


        _con.execute(
            "CREATE OR REPLACE SECRET emaps_s3 (TYPE S3, KEY_ID ?, SECRET ?, REGION ?)",
            [
                settings.aws_access_key_id,
                settings.aws_secret_access_key,
                settings.aws_region or "ap-south-1",
            ],
        )

    return _con

def print_sql_df(sql_query):
    con = get_duckdb_connection()
    
    def replace_table(match):
        keyword = match.group(1)
        schema = match.group(2)
        table = match.group(3)
        
        if schema == "bronze":
            return f"{keyword} read_parquet('{DATA_DIR}/{schema}/{table}/**/*.parquet')"
        else:
            return f"{keyword} delta_scan('{DATA_DIR}/{schema}/{table}')"

    parsed_query = re.sub(
        r'(?i)(FROM|JOIN)\s+([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)', 
        replace_table, 
        sql_query
    )
    df = con.execute(parsed_query).df()
    display(df)

In [3]:
print("Bronze Daily Flows Summary:")
print_sql_df('select * from bronze.electricity_flows limit 10')

Bronze Daily Flows Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,day,month,year
0,1777129595150,2026-04-25 20:36:35.150267+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-24 20:30:00+05:30,2026-04-25 20:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,25,04,2026


In [4]:
print("Bronze Daily Mix Summary:")
print_sql_df('select * from bronze.electricity_mix limit 10')

Bronze Daily Mix Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,day,month,year
0,1777129595150,2026-04-25 20:36:35.150267+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-24 20:30:00+05:30,2026-04-25 20:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,25,04,2026


In [5]:
print("Silver :")
print_sql_df('select process_ts, datetime, count(*) from silver.electricity_mix group by process_ts, datetime having count(*) > 1 order by process_ts, datetime limit 10')

Silver :


,process_ts,datetime,count_star()


In [6]:
print("Gold Daily Exports:")
print_sql_df('select * from gold.daily_exports limit 10')

Gold Daily Exports:


,process_ts,zone,zone_name,destination_zone,destination_zone_name,date,export_mwh,hours_covered,year,month,day
0,1777129788867,FR,France,GB,Great Britain,2026-04-25,43892.0,15,2026,4,25
1,1777129788867,FR,France,IT-NO,Italy North,2026-04-25,16548.0,15,2026,4,25
2,1777129788867,FR,France,LU,LU,2026-04-25,2760.0,15,2026,4,25
3,1777129788867,FR,France,GB,Great Britain,2026-04-24,27489.0,9,2026,4,24
4,1777129788867,FR,France,LU,LU,2026-04-24,1531.0,9,2026,4,24
5,1777129788867,FR,France,IT-NO,Italy North,2026-04-24,33987.0,9,2026,4,24


In [7]:
print("\nGold Daily Imports:")
print_sql_df("SELECT * FROM gold.daily_imports LIMIT 10")


Gold Daily Imports:


,process_ts,zone,zone_name,source_zone,source_zone_name,date,import_mwh,hours_covered,year,month,day
0,1777129788867,FR,France,DE,Germany,2026-04-25,12185.0,15,2026,4,25
1,1777129788867,FR,France,ES,Spain,2026-04-25,40187.0,15,2026,4,25
2,1777129788867,FR,France,CH,Switzerland,2026-04-25,3121.0,15,2026,4,25
3,1777129788867,FR,France,BE,Belgium,2026-04-25,38444.0,15,2026,4,25
4,1777129788867,FR,France,BE,Belgium,2026-04-24,33768.0,9,2026,4,24
5,1777129788867,FR,France,DE,Germany,2026-04-24,21702.0,9,2026,4,24
6,1777129788867,FR,France,CH,Switzerland,2026-04-24,13343.0,9,2026,4,24
7,1777129788867,FR,France,ES,Spain,2026-04-24,22780.0,9,2026,4,24


In [8]:
print("\nGold Daily Mix:")
print_sql_df("SELECT * FROM gold.daily_electricity_mix LIMIT 10")


Gold Daily Mix:


,process_ts,zone,zone_name,date,nuclear_pct,biomass_pct,wind_pct,solar_pct,hydro_pct,gas_pct,...,coal_pct,geothermal_pct,unknown_pct,total_production_mwh,fossil_free_avg_pct,renewable_avg_pct,hours_covered,year,month,day
0,1777129788867,FR,France,2026-04-24,73.854184,2.047965,8.502293,4.398373,10.493044,0.642080,...,0.0,0.0,0.0,525165.332,99.295859,25.441676,9,2026,4,24
1,1777129788867,FR,France,2026-04-25,74.282383,2.189190,3.195997,11.189395,8.409479,0.663955,...,0.0,0.0,0.0,809799.140,99.266444,24.984061,15,2026,4,25


In [9]:
print("\nPipeline State (el_state):")
print_sql_df("SELECT * FROM state.el_state ORDER BY process_ts DESC limit 10")


Pipeline State (el_state):


,process,process_ts,start_timestamp,end_timestamp,status,record_count,error_message
0,gold,1777129788867,2026-04-25 20:39:50.371386+05:30,2026-04-25 20:39:55.353971+05:30,R,16,None
1,silver,1777129719141,2026-04-25 20:38:45.969940+05:30,2026-04-25 20:38:51.653279+05:30,C,197,None
2,bronze,1777129595150,2026-04-25 20:36:35.150267+05:30,2026-04-25 20:36:44.086418+05:30,C,48,None
